# 🚀 CODI QLoRA Training on Google Colab

This notebook trains a LoRA adapter for **Qwen2.5-Coder-1.5B** to rank and explain Dockerfile optimizations.

## 📋 Requirements
- **GPU:** Colab Pro (recommended) or free tier with T4 GPU
- **VRAM:** ~8-10 GB
- **Runtime:** Python 3.10+, GPU enabled
- **Storage:** ~5 GB (model + dataset + outputs)

## ⏱️ Training Time
- **T4 GPU:** ~2-3 hours for 3 epochs
- **A100 GPU:** ~30-45 minutes

## 🔧 Setup Instructions
1. **Enable GPU:** Runtime → Change runtime type → Hardware accelerator: GPU
2. **Run cells sequentially** from top to bottom
3. **Monitor training:** Check TensorBoard logs in real-time
4. **Download adapter:** Final cell provides download link

## 📦 Outputs
- LoRA adapter weights (`adapter_model.safetensors`)
- Tokenizer files
- Training metadata
- Checkpoints (optional)


## 1️⃣ Environment Setup

First, verify GPU availability and install required dependencies.


In [ ]:
# Check GPU availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  WARNING: No GPU detected! Training will be extremely slow.")
    print("Go to: Runtime → Change runtime type → Hardware accelerator: GPU")


In [ ]:
# Install training dependencies
# This will take ~5-7 minutes on first run
!pip install -q transformers>=4.36.0 peft>=0.7.0 bitsandbytes>=0.41.0 \
    datasets>=2.16.0 accelerate>=0.25.0 tensorboard pyyaml

print("✅ Dependencies installed!")


## 2️⃣ Mount Google Drive (Optional but Recommended)

Mount Google Drive to save checkpoints and persist outputs across sessions.


In [ ]:
import os

from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create workspace in Drive
DRIVE_DIR = "/content/drive/MyDrive/codi-training"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"✅ Drive mounted at: {DRIVE_DIR}")


## 3️⃣ Upload Training Data

Upload the CODI repository ZIP file and extract it.


In [ ]:
# Upload and extract your repository
import os
import zipfile

from google.colab import files

print("📤 Upload your CODI repository ZIP file")
print("This ZIP should contain:")
print("  - codi/data/splits/train.jsonl")
print("  - codi/data/splits/val.jsonl")
print("  - codi/training/qwen15b_lora/config.yaml")
print("  - codi/training/qwen15b_lora/train.py")
print("\nClick 'Choose Files' button below...")
print()

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\nExtracting {filename}...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('/content')  # Extract to /content (ZIP contains codi/ directory)
    print("✅ Repository extracted to /content/codi/")

    print("\n📁 Verifying structure...")
    for root, dirs, files in os.walk('/content/codi', topdown=True):
        level = root.replace('/content/codi', '').count(os.sep)
        if level < 3:  # Only show first 3 levels
            indent = ' ' * 2 * level
            print(f'{indent}{os.path.basename(root)}/')
            subindent = ' ' * 2 * (level + 1)
            for file in files[:3]:  # Show first 3 files per directory
                print(f'{subindent}{file}')
        dirs[:] = [d for d in dirs if level < 2]  # Limit depth


## 3.5 Verify Extraction (Run this to check your setup!)

Verify that files were extracted correctly before proceeding.


In [ ]:
# Quick verification of file structure
import os

print("📂 Checking CODI directory structure...")
print()

expected_structure = {
    "/content/codi": "Root directory",
    "/content/codi/data/splits/train.jsonl": "Training data",
    "/content/codi/data/splits/val.jsonl": "Validation data",
    "/content/codi/training/qwen15b_lora/config.yaml": "Training config",
    "/content/codi/training/qwen15b_lora/train.py": "Training script",
    "/content/codi/patterns/rules.yml": "Optimization rules (recommended)"
}

all_good = True
for path, description in expected_structure.items():
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    print(f"{status} {description}")
    print(f"   Path: {path}")
    if exists and os.path.isfile(path):
        size = os.path.getsize(path)
        if size > 1024*1024:
            print(f"   Size: {size/(1024*1024):.1f} MB")
        else:
            print(f"   Size: {size/1024:.1f} KB")
    print()

    if not exists and "recommended" not in description:
        all_good = False

if all_good:
    print("🎉 All essential files are present!")
    print("You can proceed with training.")
else:
    print("⚠️  Some files are missing. Please check:")
    print()
    print("Common issues:")
    print("1. ZIP not uploaded (run cell 7 to upload)")
    print("2. Wrong extraction path (should extract to /content, not /content/codi)")
    print("3. Incomplete ZIP file (missing train.py or data files)")
    print()
    print("Fix: Re-run cell 7 to upload and extract your ZIP file correctly.")


## 4️⃣ Create Training Configuration

Generate the `config.yaml` file with optimized settings for Colab.


In [ ]:
# Create config.yaml optimized for Colab
config_yaml = """# CODI QLoRA Training Configuration - Colab Optimized

model:
  base_model: "Qwen/Qwen2.5-Coder-1.5B-Instruct"
  model_type: "qwen2"
  adapter_name: "qwen15b-lora-v0.1-colab"
  
  # 4-bit quantization for memory efficiency
  load_in_4bit: true
  bnb_4bit_compute_dtype: "bfloat16"
  bnb_4bit_quant_type: "nf4"
  bnb_4bit_use_double_quant: true

lora:
  r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  target_modules:
    - "q_proj"
    - "k_proj"
    - "v_proj"
    - "o_proj"
    - "gate_proj"
    - "up_proj"
    - "down_proj"
  bias: "none"
  task_type: "CAUSAL_LM"

training:
  train_data: "data/splits/train.jsonl"
  val_data: "data/splits/val.jsonl"
  
  num_train_epochs: 3
  max_steps: -1
  
  per_device_train_batch_size: 1
  per_device_eval_batch_size: 1
  gradient_accumulation_steps: 16
  
  learning_rate: 2.0e-4
  lr_scheduler_type: "cosine"
  warmup_ratio: 0.05
  
  optim: "paged_adamw_8bit"
  weight_decay: 0.01
  max_grad_norm: 1.0
  
  max_seq_length: 2048  # Reduced for Colab
  
  evaluation_strategy: "steps"
  eval_steps: 50
  save_strategy: "steps"
  save_steps: 200
  save_total_limit: 2  # Keep only last 2 checkpoints
  
  logging_steps: 10
  logging_dir: "training/qwen15b_lora/logs"
  output_dir: "training/qwen15b_lora/checkpoints"
  
  fp16: false
  bf16: true
  
  seed: 42
  dataloader_num_workers: 2
  remove_unused_columns: false
  group_by_length: false

hardware:
  min_vram_gb: 8
  recommended_vram_gb: 16

dataset:
  instruction_field: "instruction"
  input_field: "input"
  output_field: "output"
  shuffle_train: true
  shuffle_seed: 42

adapter:
  output_path: "models/adapters/qwen15b-lora-v0.1-colab"
  files:
    - "adapter_config.json"
    - "adapter_model.bin"
    - "adapter_model.safetensors"
    - "metadata.json"
  metadata:
    version: "v0.1-colab"
    base_model: "Qwen/Qwen2.5-Coder-1.5B-Instruct"
  checksum_algo: "sha256"

export:
  enable_gguf_export: false
  enable_merge: false

validation:
  dry_run_supported: true
"""

# Write config to file
with open("/content/codi/training/qwen15b_lora/config.yaml", "w") as f:
    f.write(config_yaml)

print("✅ Configuration saved to: /content/codi/training/qwen15b_lora/config.yaml")
print("\\nKey settings:")
print("  - Model: Qwen2.5-Coder-1.5B (4-bit)")
print("  - LoRA rank: 16")
print("  - Batch size: 1 (effective: 16 with grad accumulation)")
print("  - Max sequence length: 2048 tokens")
print("  - Epochs: 3")


## 6️⃣ Validate Environment (Dry Run)

Check that everything is configured correctly before starting training.


In [ ]:
# Run environment validation
import os

print("🔍 Checking file structure...")
print()

# Check if key files exist
required_files = [
    "/content/codi/data/splits/train.jsonl",
    "/content/codi/data/splits/val.jsonl",
    "/content/codi/training/qwen15b_lora/config.yaml",
    "/content/codi/training/qwen15b_lora/train.py"
]

all_exist = True
for file_path in required_files:
    exists = os.path.exists(file_path)
    status = "✓" if exists else "❌"
    print(f"  {status} {file_path.replace('/content/codi/', '')}")
    if not exists:
        all_exist = False

print()

if not all_exist:
    print("❌ Some required files are missing!")
    print("Please make sure you:")
    print("  1. Uploaded the correct ZIP file")
    print("  2. Extracted it to /content (not /content/codi)")
    print("  3. The ZIP contains the codi/ directory structure")
    print()
    print("If you used Option B (sample data), train.py won't be available.")
    print("You can proceed to cell 16 for inline training.")
else:
    print("✅ All required files found!")
    print()
    print("Running dry-run validation...")
    print()
    os.chdir("/content/codi")
    !python /content/codi/training/qwen15b_lora/train.py --config training/qwen15b_lora/config.yaml --dry-run


## 7️⃣ Start Training! 🚀

This cell will train the LoRA adapter. Expect ~2-3 hours on T4 GPU.


In [ ]:
%%time
# Start training
import json
import os

import torch
import yaml
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

os.chdir("/content/codi")

# Load config
with open("training/qwen15b_lora/config.yaml") as f:
    config = yaml.safe_load(f)

print("="*60)
print("🚀 CODI QLoRA Training Starting")
print("="*60)

# 1. Load datasets
print("\n📊 Loading datasets...")
train_dataset = load_dataset("json", data_files="data/splits/train.jsonl", split="train")
val_dataset = load_dataset("json", data_files="data/splits/val.jsonl", split="train")
print(f"  ✓ Train examples: {len(train_dataset)}")
print(f"  ✓ Val examples: {len(val_dataset)}")

# 2. Load model with 4-bit quantization
print("\n🤖 Loading model...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    config["model"]["base_model"],
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print(f"  ✓ Model loaded: {config['model']['base_model']}")

# 3. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    config["model"]["base_model"],
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id
print("  ✓ Tokenizer configured")

# 4. Apply LoRA
print("\n🔧 Applying LoRA...")
lora_config = LoraConfig(
    r=config["lora"]["r"],
    lora_alpha=config["lora"]["lora_alpha"],
    lora_dropout=config["lora"]["lora_dropout"],
    target_modules=config["lora"]["target_modules"],
    bias=config["lora"]["bias"],
    task_type=config["lora"]["task_type"],
)
model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"  ✓ Trainable params: {trainable_params:,} ({100*trainable_params/total_params:.2f}%")

# 5. Format and tokenize
def format_instruction(instruction, input_text, output_text):
    """Format a single training example into instruction prompt."""
    return f"""### Instruction:
{instruction}

### Input:
{input_text}

### Output:
{output_text}"""

def preprocess(examples):
    """Preprocess batched examples for training.

    When batched=True, examples is a dict of lists:
    {"instruction": [...], "input": [...], "output": [...]}
    """
    texts = [
        format_instruction(inst, inp, out)
        for inst, inp, out in zip(
            examples['instruction'],
            examples['input'],
            examples['output']
        )
    ]
    model_inputs = tokenizer(
        texts,
        max_length=config["training"]["max_seq_length"],
        truncation=True,
        padding="max_length",
    )
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

print("\n📝 Tokenizing datasets...")
tokenized_train = train_dataset.map(preprocess, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess, batched=True, remove_columns=val_dataset.column_names)
print("  ✓ Tokenization complete")

# 6. Training arguments
training_args = TrainingArguments(
    output_dir=config["training"]["output_dir"],
    num_train_epochs=config["training"]["num_train_epochs"],
    per_device_train_batch_size=config["training"]["per_device_train_batch_size"],
    per_device_eval_batch_size=config["training"]["per_device_eval_batch_size"],
    gradient_accumulation_steps=config["training"]["gradient_accumulation_steps"],
    learning_rate=config["training"]["learning_rate"],
    lr_scheduler_type=config["training"]["lr_scheduler_type"],
    warmup_ratio=config["training"]["warmup_ratio"],
    weight_decay=config["training"]["weight_decay"],
    max_grad_norm=config["training"]["max_grad_norm"],
    optim=config["training"]["optim"],
    # evaluation_strategy=config["training"]["evaluation_strategy"], # Temporarily commented out
    # eval_steps=config["training"]["eval_steps"],                 # Temporarily commented out
    # save_strategy=config["training"]["save_strategy"],           # Temporarily commented out
    # save_steps=config["training"]["save_steps"],                 # Temporarily commented out
    # save_total_limit=config["training"]["save_total_limit"],     # Temporarily commented out
    logging_steps=config["training"]["logging_steps"],
    logging_dir=config["training"]["logging_dir"],
    fp16=False,
    bf16=True,
    seed=config["training"]["seed"],
    report_to="tensorboard",
)

# 7. Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

print("\n" + "="*60)
print("🔥 Training Started!")
print("="*60)
print(f"  Epochs: {config['training']['num_train_epochs']}")
print(f"  Effective batch size: {config['training']['gradient_accumulation_steps']}")
print(f"  Learning rate: {config['training']['learning_rate']}")
print(f"  Checkpoints: {config['training']['output_dir']}")
print("="*60 + "\n")

# 8. Train!
trainer.train()

print("\n" + "="*60)
print("✅ Training Complete!")
print("="*60)


In [ ]:
# Save adapter
import os
from datetime import datetime
from pathlib import Path

adapter_dir = Path(config["adapter"]["output_path"])
adapter_dir.mkdir(parents=True, exist_ok=True)

# Save model and tokenizer
print("💾 Saving adapter...")
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"  ✓ Adapter saved to: {adapter_dir}")

# Generate metadata
metadata = {
    "version": "v0.1-colab",
    "base_model": config["model"]["base_model"],
    "adapter_name": config["model"]["adapter_name"],
    "training_date": datetime.utcnow().isoformat() + "Z",
    "platform": "Google Colab",
    "hyperparameters": {
        "lora_r": config["lora"]["r"],
        "lora_alpha": config["lora"]["lora_alpha"],
        "lora_dropout": config["lora"]["lora_dropout"],
        "num_epochs": config["training"]["num_train_epochs"],
        "learning_rate": config["training"]["learning_rate"],
        "effective_batch_size": config["training"]["gradient_accumulation_steps"],
    },
    "dataset": {
        "num_train_examples": len(train_dataset),
        "num_val_examples": len(val_dataset),
    }
}

metadata_path = adapter_dir / "metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"  ✓ Metadata saved to: {metadata_path}")

# List saved files
print("\\n📁 Saved files:")
for file in sorted(adapter_dir.glob("*")):
    size_mb = file.stat().st_size / 1e6
    print(f"  - {file.name} ({size_mb:.1f} MB)")


In [ ]:
# Test inference
test_prompt = """### Instruction:
Optimize this Python Dockerfile for production deployment.

### Input:
FROM python:3.12
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
CMD python app.py

### Output:"""

print("🧪 Testing adapter with sample prompt...\\n")
print("-" * 60)
print(test_prompt)
print("-" * 60)

# Generate response
inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract only the generated part (after "### Output:")
generated = response.split("### Output:")[-1].strip()
print("\\n🤖 Generated response:")
print("-" * 60)
print(generated)
print("-" * 60)
print("\\n✅ Inference test complete!")


## 🔟 Copy to Google Drive

Save the adapter to Google Drive for persistence.


In [ ]:
# Copy adapter to Google Drive
import shutil

drive_adapter_dir = f"{DRIVE_DIR}/adapter-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
shutil.copytree(adapter_dir, drive_adapter_dir)

print("✅ Adapter copied to Google Drive:")
print(f"   {drive_adapter_dir}")
print("\\nThis ensures your adapter persists even if the Colab session ends.")


## 📦 Download Adapter (ZIP)

Create a ZIP file and download it to your local machine.


In [ ]:
# Create ZIP archive
import zipfile

from google.colab import files

zip_filename = f"qwen15b-lora-adapter-{datetime.now().strftime('%Y%m%d-%H%M%S')}.zip"
zip_path = f"/content/{zip_filename}"

print(f"📦 Creating ZIP archive: {zip_filename}")
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in adapter_dir.rglob("*"):
        if file.is_file():
            arcname = file.relative_to(adapter_dir.parent)
            zipf.write(file, arcname)
            print(f"  + {arcname}")

zip_size_mb = Path(zip_path).stat().st_size / 1e6
print(f"\\n✅ ZIP created: {zip_filename} ({zip_size_mb:.1f} MB)")

# Download
print("\\n⬇️  Starting download...")
files.download(zip_path)
print("\\n✅ Download complete! Check your browser's download folder.")


## 📊 View TensorBoard Logs (Optional)

Monitor training metrics with TensorBoard.


In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Launch TensorBoard
%tensorboard --logdir /content/codi/training/qwen15b_lora/logs

print("📊 TensorBoard launched! View training metrics above.")


## 🎉 Summary and Next Steps

Your LoRA adapter has been trained successfully! Here's what you have:

### 📦 Adapter Files
- `adapter_model.safetensors` - LoRA weights (safe format)
- `adapter_config.json` - LoRA configuration
- `tokenizer.json` - Tokenizer vocabulary
- `metadata.json` - Training metadata

### 📈 Typical Results
- **Adapter size:** ~50-100 MB
- **Training time:** 2-3 hours on T4
- **Train loss:** Should decrease to <1.5
- **Val loss:** Should be close to train loss (no overfitting)

### 🚀 Integration with CODI

1. **Extract the ZIP** to your CODI repository:
   ```bash
   unzip qwen15b-lora-adapter-*.zip -d models/adapters/
   ```

2. **Verify adapter location:**
   ```bash
   ls models/adapters/qwen15b-lora-v0.1-colab/
   ```

3. **Rebuild CODI Complete container:**
   ```bash
   make build-complete
   ```

4. **Test with LLM endpoints:**
   ```bash
   docker run -v $PWD:/work codi:complete codi llm rank \\
     --candidates runs/<timestamp>/candidates/*.Dockerfile
   ```

### 🔧 Troubleshooting

**If training OOMed:**
- Reduce `max_seq_length` to 1024
- Increase `gradient_accumulation_steps` to 32
- Use `per_device_train_batch_size: 1`

**If loss isn't decreasing:**
- Check data quality
- Increase `warmup_ratio` to 0.1
- Lower `learning_rate` to 1e-4

**Need help?** Check the README at `training/qwen15b_lora/README.md`

---

**🎊 Congratulations on training your CODI adapter!**
